# Musanga — Optimize UAV GeoTIFFs for direct Earth Engine Asset upload

**Colab/Drive:** `kristof.vanoost@uclouvain.be` (Colab Pro). **Earth Engine project:** `musanga-511110`.

**No Google Cloud Storage bucket, no staging, and no storage-service billing.** This notebook optimizes GeoTIFFs and saves them in your university Google Drive. You then upload files **directly from your computer** using [Earth Engine Code Editor → Assets → NEW → Image upload](https://code.earthengine.google.com/).

**Important:** The Code Editor's file picker selects **local files**, not paths in the Colab runtime. Download the optimized TIFFs from Google Drive to your computer (or use Google Drive for desktop) before selecting them in the Code Editor. For a very large file, use Drive for desktop rather than a browser download.

**Raster strategy:** Native pixel resolution, losslessly compressed RGB with 4th alpha band to preserve transparency (if needed); probability `uint16` 0–10000 → `uint8` 0–200 (0.5 percentage-point increments), NoData 255. Binary overlay is computed dynamically from probability, so only **two assets per site**. Original files stay unchanged.

**Upload limit:** Earth Engine's Code Editor supports GeoTIFFs up to **10 GB per file**. Inspect resulting TIFF sizes before attempting uploads. Optimize one site (Baego) first.

**Caution:** The U-Net probability GeoTIFF can contain ROI-only valid data. The probability and RGB extents may differ; do not force them to the same grid.

**Expected final assets:** `projects/musanga-511110/assets/musanga_uav/Baego_RGB` and `.../Baego_Musanga_probability` (similarly for the other sites). Use **Mean** pyramiding; for RGB select **Use last band as alpha band**, for probability set **No-data value = 255** in upload Advanced options.

[Updated GEE viewer](https://github.com/kvanoost/Musanga/blob/main/gee/Musanga_UAV_probability_viewer.js)


In [ ]:
# Install only the packages needed for raster processing and optional EE read-only checks.
%pip -q install "google-auth==2.49.0" earthengine-api rasterio


In [ ]:
from pathlib import Path
import ee, rasterio
from google.colab import auth, drive

CLOUD_PROJECT="musanga-511110"
ASSET_FOLDER=f"projects/{CLOUD_PROJECT}/assets/musanga_uav"
auth.authenticate_user()
ee.Initialize(project=CLOUD_PROJECT)
print("Earth Engine check:",ee.Number(42).getInfo())
drive.mount("/content/drive")
ROOT=Path("/content/drive/MyDrive/Colab Notebooks/Musanga")
GEO=ROOT/"data"/"Geo_Ref"
MAPS=ROOT/"KVO_work"/"KVO_04_final_multisite"/"probability_maps"
# Saved on Drive to be downloaded to your computer for browser-based Earth Engine upload.
OPT_DIR=ROOT/"KVO_work"/"KVO_07_GEE_direct_upload_optimized"
OPT_DIR.mkdir(parents=True,exist_ok=True)
print("Project:",CLOUD_PROJECT,"\nOptimized outputs:",OPT_DIR)


In [ ]:
# Enable Baego first; add sites when ready.
SELECTED_SITES = ["Baego"]
# SELECTED_SITES = ["Baego", "Yoko", "Yangambi", "MOS_1_2"]

SITE_RASTERS = {
    "Baego": "Baego_georef.tif",
    "Yoko": "Yoko_kvo_georef.tif",
    "Yangambi": "Yangambi_georef.tif",
    "MOS_1_2": "MOS_1_2_kvo.tif",
}

def inspect_site(site):
    rgb=GEO/SITE_RASTERS[site]
    prob=MAPS/f"{site}_musanga_probability_uint16.tif"
    for label,path in [("RGB",rgb),("probability",prob)]:
        if not path.exists():
            raise FileNotFoundError(f"{site} {label} missing: {path}")
        with rasterio.open(path) as src:
            print(site,label, "bands",src.count,"dtype",src.dtypes,"CRS",src.crs,
                  "resolution",src.res,"NoData",src.nodata,
                  "size GB",round(path.stat().st_size/1e9,2))
            assert src.crs is not None, f"Missing CRS: {path}"
            if label=="RGB":
                assert src.count>=3, f"RGB needs at least 3 bands: {path}"
            else:
                assert src.count==1 and src.dtypes[0]=="uint16", f"Expected one uint16 probability band: {path}"
    return rgb,prob

SOURCES={site:inspect_site(site) for site in SELECTED_SITES}
print("Inputs verified. Next: run the optimization cell BEFORE staging.")

## Block 4 — Native-resolution raster optimization for direct Assets upload

Outputs are written to Google Drive. RGB is **lossless 4-band RGBA** to preserve transparency in the Code Editor's “Use last band as alpha band” option. Probability is 8-bit with 255 reserved for NoData. The notebook validates sample windows and compares output sizes. Large raster processing on mounted Drive may take time.


In [ ]:
# BLOCK 4 — Optimize all selected rasters BEFORE GCS upload (windowed, low memory)
# Keeps RGB at native pixel size, first 3 bands, losslessly compressed.
# Converts probability 0..10000 uint16 -> 0..200 uint8; 255=NoData; binary stays virtual.
from pathlib import Path
import numpy as np, rasterio
from rasterio.windows import Window
from rasterio.enums import ColorInterp

OPT_DIR=ROOT/"KVO_work"/"KVO_07_GEE_direct_upload_optimized"
OPT_DIR.mkdir(parents=True,exist_ok=True)
OPTIMIZED={}
P_SCALE=200   # 0.5 percentage-point bins, threshold in GEE uses /200
P_NODATA=255

def windows(width,height,block=512):
    for row in range(0,height,block):
        for col in range(0,width,block):
            yield Window(col,row,min(block,width-col),min(block,height-row))

def optimize_rgb(inp,out):
    with rasterio.open(inp) as src:
        assert src.count>=3, f"RGB needs >=3 bands: {inp}"
        assert src.dtypes[0]=="uint8", (
            f"RGB is {src.dtypes[0]}; refusing automatic 16-bit stretch. "
            "Choose visualization stretch explicitly before uint8 conversion.")
        profile=src.profile.copy()
        profile.update(driver="GTiff",count=4,dtype="uint8",nodata=None,
                       compress="DEFLATE",predictor=2,tiled=True,
                       blockxsize=512,blockysize=512,interleave="pixel",BIGTIFF="IF_SAFER")
        with rasterio.open(out,"w",**profile) as dst:
            dst.colorinterp=(ColorInterp.red,ColorInterp.green,ColorInterp.blue,ColorInterp.alpha)
            has_alpha=(src.count>=4 and src.colorinterp[3]==ColorInterp.alpha)
            for w in windows(src.width,src.height):
                rgb=src.read([1,2,3],window=w)
                # Preserve band-level validity and alpha transparency while dropping alpha band.
                validity=np.logical_and.reduce(src.read_masks([1,2,3],window=w)>0,axis=0)
                if has_alpha: validity &= src.read(4,window=w)>0
                dst.write(rgb,window=w,indexes=[1,2,3])
                dst.write((validity*255).astype("uint8"),4,window=w)
    return out

def optimize_probability(inp,out):
    with rasterio.open(inp) as src:
        assert src.count==1 and src.dtypes[0]=="uint16"
        profile=src.profile.copy()
        profile.update(driver="GTiff",count=1,dtype="uint8",nodata=P_NODATA,
                       compress="DEFLATE",predictor=2,tiled=True,
                       blockxsize=512,blockysize=512,BIGTIFF="IF_SAFER")
        with rasterio.open(out,"w",**profile) as dst:
            dst.set_band_description(1,"probability_uint8_scale_200")
            for w in windows(src.width,src.height):
                values=src.read(1,window=w)
                valid=(src.read_masks(1,window=w)>0)&(values!=65535)
                if np.any(valid):
                    assert np.max(values[valid])<=10000, "Unexpected probability >10000"
                scaled=np.full(values.shape,P_NODATA,dtype="uint8")
                scaled[valid]=np.rint(values[valid].astype("float32")*(P_SCALE/10000)).clip(0,P_SCALE).astype("uint8")
                dst.write(scaled,1,window=w)
    return out

def quick_validate(source,target,kind):
    with rasterio.open(source) as a, rasterio.open(target) as b:
        assert a.width==b.width and a.height==b.height and a.crs==b.crs
        assert a.transform.almost_equals(b.transform), "GeoTransform differs"
        if kind=="RGB":
            assert b.count==4 and b.dtypes[0]=="uint8"
        else:
            assert b.count==1 and b.dtypes[0]=="uint8" and b.nodata==255
        # Sample spatially separated windows, including corners.
        coords=[(0,0),(max(0,a.width//2-128),max(0,a.height//2-128)),
                (max(0,a.width-256),max(0,a.height-256))]
        for col,row in coords:
            w=Window(col,row,min(256,a.width-col),min(256,a.height-row))
            if kind=="probability":
                inp=a.read(1,window=w)
                out=b.read(1,window=w)
                ok=(a.read_masks(1,window=w)>0)&(inp!=65535)
                assert np.all(out[~ok]==255), "NoData mismatch"
                if ok.any():
                    err=np.abs(out[ok].astype("float32")/P_SCALE-inp[ok].astype("float32")/10000)
                    assert err.max()<=0.002501, f"Quantization error {err.max()}"
            else:
                assert np.array_equal(a.read([1,2,3],window=w),b.read([1,2,3],window=w))
    print(kind, "validation PASS",f"{source.stat().st_size/1e9:.2f} -> {target.stat().st_size/1e9:.2f} GB",
          f"({target.stat().st_size/source.stat().st_size:.0%} of original)")

for site,(rgb,prob) in SOURCES.items():
    rgb_out=OPT_DIR/f"{site}_RGB_optimized.tif"
    prob_out=OPT_DIR/f"{site}_Musanga_probability_uint8_optimized.tif"
    # Reuse derivatives when present; validation is still repeated.
    if not rgb_out.exists(): optimize_rgb(rgb,rgb_out)
    assert not Path(str(rgb_out)+".msk").exists(), "Unexpected external mask detected"
    quick_validate(rgb,rgb_out,"RGB")
    if not prob_out.exists(): optimize_probability(prob,prob_out)
    quick_validate(prob,prob_out,"probability")
    OPTIMIZED[site]={"RGB":rgb_out,"Musanga_probability":prob_out}
print("READY for direct GEE Asset upload:",OPTIMIZED)
print("Original source rasters on Drive were not modified.")


## Block 5 — Upload instructions (manual; no GCS)

1. Look at the sizes reported by Block 4. Earth Engine's browser uploader supports GeoTIFF files **up to 10 GB each**.
2. Download the optimized `.tif` files from Drive to your **computer**, or access them via **Google Drive for desktop**. Colab's `/content/drive/` path is not available in the Code Editor file picker.
3. Open [Earth Engine Code Editor](https://code.earthengine.google.com/) using the account with Musanga asset-write permissions. Select Cloud project `musanga-511110`.
4. In **Assets**, create an image asset directly through **NEW → Image upload**. Choose the local optimized TIFF and enter the matching asset ID printed in Block 5 below. Repeat once for RGB and probability.
5. Under **Advanced**, choose **Mean** pyramiding. **RGB:** *Use last band as alpha band* (4th band). **Probability:** *No-data value* `255`.
6. Watch **Tasks** until ingestion completes. Then open the [Musanga GEE viewer](https://github.com/kvanoost/Musanga/blob/main/gee/Musanga_UAV_probability_viewer.js).

There is **no separate binary asset**. For now the viewer interprets probability integers `0..200` as `0..1`, with `255` as NoData.

If an individual optimized TIFF exceeds 10 GB, stop and consider tiling or a different upload method rather than assuming the Code Editor will accept it.


In [ ]:
# BLOCK 5 — Print exact filenames and intended Earth Engine asset IDs.
assert OPTIMIZED,"Run Block 4 first"
for site,items in OPTIMIZED.items():
    for kind,path in items.items():
        print("\nLOCAL / GOOGLE DRIVE:",path)
        print("FILE SIZE GB:",round(path.stat().st_size/1e9,3))
        print("EARTH ENGINE ASSET:",f"{ASSET_FOLDER}/{site}_{kind}")
        print("UPLOAD MASK:", "Use last band as alpha band" if kind=="RGB" else "No-data value = 255")
        if path.stat().st_size > 10_000_000_000:
            print("WARNING: larger than Code Editor's 10 GB upload limit")
print("\nUse local file selection in the browser Code Editor. This notebook does not upload assets.")


## Block 6 — Optional read-only confirmation after browser uploads

Run after the Code Editor tasks finish to check whether the assets are visible. This cell does not create or change Earth Engine assets.


In [ ]:
# BLOCK 6 — Read-only asset status check.
for site in SELECTED_SITES:
    for kind in ("RGB","Musanga_probability"):
        name=f"{ASSET_FOLDER}/{site}_{kind}"
        try:
            asset=ee.data.getAsset(name)
            print("READY:",name,"type:",asset.get("type"))
        except Exception as exc:
            print("NOT AVAILABLE:",name,"—",str(exc)[:250])
